# National League Team Trends — Data Visualization Notebook

Comprehensive analysis and visualization of NL historical performance data (1876–2025).
Sources: Baseball Reference, Retrosheet, SABR Lahman Database, Baseball Almanac

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab20')

# Load data
df = pd.read_csv('data/nl_historical_data.csv')
franchise = pd.read_csv('data/franchise_summary.csv')
champs = pd.read_csv('nl_champions.csv')
wins = pd.read_csv('nl_team_wins.csv')
eras = pd.read_csv('data/era_analysis.csv')
comprehensive = pd.read_csv('data/nl_comprehensive_performance.csv')

print(f'Historical data: {len(df)} rows, {df["season"].min()}–{df["season"].max()}')
print(f'Franchise summary: {len(franchise)} franchises')
print(f'Champions data: {len(champs)} seasons')
print(f'Comprehensive data: {len(comprehensive)} rows')

## 1. All-Time Franchise Win-Loss Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top franchises by all-time wins
top_franchises = franchise.sort_values('win_pct', ascending=False).head(8)
colors = ['#2D7D4F', '#3A6EA5', '#C8102E', '#FFB000', '#000000', '#0E3386', '#1D428A', '#183025']

axes[0].barh(top_franchises['franchise'], top_franchises['ws_titles'], color=colors[:len(top_franchises)])
axes[0].set_xlabel('World Series Titles')
axes[0].set_title('World Series Titles by Franchise')

# Win% vs Total Games
axes[1].scatter(franchise['seasons'], franchise['win_pct'] * 100, s=franchise['seasons']/3, c=colors, alpha=0.8)
axes[1].set_xlabel('Seasons Played')
axes[1].set_ylabel('Win Percentage (%)')
axes[1].set_title('Win % vs Longevity')

for _, row in franchise.iterrows():
    axes[1].annotate(row['franchise'].split()[-1], (row['seasons'], row['win_pct']*100), fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/figures/franchise_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. ERA Performance Heatmap

In [ ]:
# Pivot table: Team × Season win%
pivot = df.pivot_table(values='win_pct', index='modern_name', columns='season', aggfunc='mean')

fig, ax = plt.subplots(figsize=(20, 10))
sns.heatmap(pivot, cmap='RdYlGn', center=0.5, ax=ax, 
            cbar_kws={'label': 'Win Percentage'}, xticklabels=False)
ax.set_title('NL Franchise Win Percentage Heatmap (1876–2025)', fontsize=14)
ax.set_ylabel('Franchise')
plt.tight_layout()
plt.savefig('notebooks/figures/era_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Dynasty Detection — Rolling Win Averages

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

teams = ['St. Louis Cardinals', 'San Francisco Giants', 'Los Angeles Dodgers', 
         'Chicago Cubs', 'Atlanta Braves', 'Pittsburgh Pirates']

palette = sns.color_palette('tab10', n_colors=len(teams))

for i, team in enumerate(teams):
    team_data = df[df['modern_name'] == team].sort_values('season').copy()
    team_data['rolling_5yr'] = team_data['win_pct'].rolling(5).mean()
    team_data['rolling_10yr'] = team_data['win_pct'].rolling(10).mean()
    
    ax.plot(team_data['season'], team_data['rolling_5yr'], label=f'{team} (5yr)', 
            color=palette[i], alpha=0.7, linewidth=2)
    ax.plot(team_data['season'], team_data['rolling_10yr'], label=f'{team} (10yr)', 
            color=palette[i], alpha=0.3, linewidth=2, linestyle='--')

ax.set_xlabel('Season')
ax.set_ylabel('Rolling Win Percentage')
ax.set_title('NL Franchise Dynasty Detection — 5-Year & 10-Year Rolling Win Averages', fontsize=14)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_ylim(0.4, 0.8)

plt.tight_layout()
plt.savefig('notebooks/figures/dynasty_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. NL Championship Drought Tracker

In [ ]:
# Calculate years between championships
ws_champions = champs[champs['Champion'] != '???'].dropna(subset=['Champion'])

fig, ax = plt.subplots(figsize=(12, 6))

for team in df['modern_name'].unique():
    team_ws = ws_champions[ws_champions['Champion'] == team].copy()
    if len(team_ws) > 1:
        gaps = team_ws['Year'].diff().iloc[1:]
        ax.scatter(range(len(gaps)), gaps, label=team, s=50, alpha=0.8)

ax.set_xlabel('Championship Sequence')
ax.set_ylabel('Years Between Championships')
ax.set_title('NL Championship Droughts by Franchise', fontsize=14)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/figures/championship_droughts.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Division Dominance Over Time

In [ ]:
# Count division winners by decade and division
comprehensive['decade'] = (comprehensive['season'] // 10) * 10
divisional_winners = comprehensive[comprehensive['division'] != 'N/A'].groupby(
    ['decade', 'division']
).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=divisional_winners, x='decade', y='count', hue='division', ax=ax)
ax.set_xlabel('Decade')
ax.set_ylabel('Division Titles')
ax.set_title('NL Division Title Winners by Decade and Division', fontsize=14)

plt.tight_layout()
plt.savefig('notebooks/figures/division_dominance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Run Differential Trend Lines

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Run differential over time for top franchises
top_teams = ['St. Louis Cardinals', 'San Francisco Giants', 'Los Angeles Dodgers', 
             'Chicago Cubs', 'Atlanta Braves']

colors = sns.color_palette('tab10', n_colors=5)

for i, team in enumerate(top_teams):
    team_data = comprehensive[comprehensive['modern_name'] == team].sort_values('season')
    team_data = team_data.dropna(subset=['run_differential'])
    axes[0].plot(team_data['season'], team_data['run_differential'], 
                 label=team.split()[-1], color=colors[i], linewidth=2)

axes[0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Run Differential')
axes[0].set_title('Run Differential Trends', fontsize=14)
axes[0].legend(fontsize=9)

# Win % by era
era_data = comprehensive.groupby('era').agg({
    'win_pct': 'mean',
    'runs_scored': 'mean',
    'runs_allowed': 'mean'
}).reset_index()

x = np.arange(len(era_data))
width = 0.25

axes[1].bar(x - width, era_data['runs_scored'], width, label='RS', alpha=0.8)
axes[1].bar(x, era_data['runs_allowed'], width, label='RA', alpha=0.8)
axes[1].bar(x + width, era_data['win_pct'] * 100, width, label='Win%', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(era_data['era'], rotation=45, fontsize=9)
axes[1].set_ylabel('Value')
axes[1].set_title('Era Performance Comparison', fontsize=14)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('notebooks/figures/run_differential.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Head-to-Head Win Matrix (Franchise vs. Franchise)

In [ ]:
# Create H2H matrix from comprehensive data
teams_list = sorted(comprehensive['modern_name'].unique())
h2h_matrix = pd.DataFrame(0, index=teams_list, columns=teams_list)

# Approximate H2H from season records (not perfect but illustrative)
for i in range(len(teams_list)):
    for j in range(len(teams_list)):
        if i != j:
            team_totals = comprehensive[comprehensive['modern_name'] == teams_list[i]]
            opponent_totals = comprehensive[comprehensive['modern_name'] == teams_list[j]]
            h2h_matrix.iloc[i, j] = team_totals['wins'].sum() - opponent_totals['losses'].sum()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(h2h_matrix.astype(float), annot=True, fmt='.0f', cmap='RdYlGn', 
            center=0, ax=ax, 
            xticklabels=[t.split()[-1] for t in teams_list],
            yticklabels=[t.split()[-1] for t in teams_list], 
            linewidths=0.5)
ax.set_title('NL Franchise Head-to-Head Win Matrix (1876–2025)', fontsize=14)
ax.set_ylabel('Franchise')
ax.set_xlabel('Opponent Franchise')

plt.tight_layout()
plt.savefig('notebooks/figures/h2h_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Expansion Impact Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Win% by era
era_order = ['Pre-Modern', 'Dead Ball', 'Live Ball', 'Post-War', 'Expansion', 'Divisional', 'COVID-Modern']
era_data = comprehensive.groupby('era').agg({'win_pct': 'mean', 'wins': 'mean', 'losses': 'mean'}).reindex(era_order)

x = np.arange(len(era_data))
axes[0].bar(x, era_data['win_pct'] * 100, color=sns.color_palette('Blues', len(era_data)), alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(era_data.index, rotation=45, fontsize=9)
axes[0].set_ylabel('Average Win Percentage (%)')
axes[0].set_title('NL Average Win% by Era', fontsize=14)

# Number of teams by era
era_teams = comprehensive.groupby('era')['modern_name'].nunique().reindex(era_order)
axes[1].bar(x, era_teams, color=sns.color_palette('Oranges', len(era_order)), alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(era_order, rotation=45, fontsize=9)
axes[1].set_ylabel('Number of Teams')
axes[1].set_title('NL Number of Teams by Era', fontsize=14)

plt.tight_layout()
plt.savefig('notebooks/figures/expansion_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Recent Season Performance (2015–2025)

In [ ]:
recent = wins[wins['Year'] >= 2015].copy()

fig, ax = plt.subplots(figsize=(14, 7))

teams_to_plot = ['Atlanta Braves', 'Los Angeles Dodgers', 'Chicago Cubs', 
                 'San Francisco Giants', 'St. Louis Cardinals', 'Philadelphia Phillies']

colors = sns.color_palette('tab10', n_colors=len(teams_to_plot))

for i, team in enumerate(teams_to_plot):
    team_data = recent[recent['Team'] == team].sort_values('Year')
    ax.plot(team_data['Year'], team_data['WinPct'] * 100, 
            marker='o', linewidth=2, label=team.split()[-1], color=colors[i])

ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='.500')
ax.set_xlabel('Season')
ax.set_ylabel('Win Percentage (%)')
ax.set_title('NL Team Win % — Recent Years (2015–2025)', fontsize=14)
ax.legend(fontsize=9)
ax.set_xlim(2014.5, 2025.5)

plt.tight_layout()
plt.savefig('notebooks/figures/recent_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Selected Key Quick Stats

In [ ]:
print('=== NL All-Time Franchise Leaders ===')
print(franchise[['franchise','win_pct','ws_titles','pennants','seasons']].sort_values('win_pct', ascending=False).to_string(index=False))
print()
print('=== NL Best Single-Season Records ===')
best_records = comprehensive.nlargest(10, 'win_pct')[['season','modern_name','wins','losses','win_pct','notable_milestones']]
print(best_records.to_string(index=False))
print()
print('=== NL Era Summary ===')
print(eras.to_string(index=False))
print()
print('=== NL Champions by Decade ===')
champs['decade'] = (champs['Year'] // 10) * 10
champ_counts = champs.groupby('decade')['Champion'].count()
print(champ_counts.to_string())